# 🎮 Prompt Engineering Playground

**Estimated time: 10-15 minutes**

## Welcome to the Art and Science of Prompt Engineering!

### Why Prompt Engineering Matters

Prompt engineering is the skill of crafting effective instructions for Large Language Models (LLMs). Think of it as learning to communicate with a highly intelligent but literal-minded assistant:

- 🎯 **Better prompts = Better outputs**: The difference between vague and specific prompts can be dramatic
- 💰 **Cost efficiency**: Good prompts get you the right answer faster (fewer tokens, fewer retries)
- 🚀 **Unlock capabilities**: Techniques like few-shot learning and chain-of-thought can solve complex problems
- 🏆 **Competitive advantage**: In the real world, prompt engineering skills are highly valued

### What You'll Learn

1. **Specificity**: How precise instructions improve outputs
2. **Few-shot learning**: Teaching by example
3. **Chain-of-thought**: Getting models to "think out loud"
4. **System vs User prompts**: Setting the stage vs giving instructions
5. **Structured extraction**: Getting clean, usable data from messy text

### 🎲 Interactive Challenges

Throughout this notebook, you'll compete to craft the best prompts. Can you beat the examples provided?

## 🔧 Setup: Choose Your LLM Provider

This notebook supports multiple backends:
- **OpenAI** (GPT-3.5/GPT-4) - Fast, powerful, paid API
- **Anthropic** (Claude) - Great reasoning, paid API
- **Hugging Face** (Free models) - Free, runs locally or via API
- **Mock Mode** - No API needed, uses pre-generated responses for learning

### API Key Setup (Optional)

If you have API keys, set them as environment variables:

```bash
# In your terminal:
export OPENAI_API_KEY="your-key-here"
export ANTHROPIC_API_KEY="your-key-here"
export HUGGINGFACE_API_KEY="your-key-here"  # Optional for HF
```

Or create a `.env` file in the project root:
```
OPENAI_API_KEY=your-key-here
ANTHROPIC_API_KEY=your-key-here
HUGGINGFACE_API_KEY=your-key-here
```

**Don't have API keys? No problem!** We'll use mock responses or local models.

In [4]:
if os.getenv("ANTHROPIC_API_KEY"):
    print("🔑 Detected Anthropic API key")

In [6]:
from dotenv import load_dotenv
load_dotenv()  # loads ANTHROPIC_API_KEY from .env

In [11]:
! echo $ANTHROPIC_API_KEY

In [9]:
! echo $ANTHROPIC_API_KEY
if os.getenv("ANTHROPIC_API_KEY"):
    print("🔑 Detected Anthropic API key")

In [1]:
# Install required packages (uncomment if needed)
# !pip install openai anthropic transformers huggingface_hub python-dotenv

import os
from typing import Optional, List, Dict, Any
import json
from datetime import datetime

# Try to load environment variables
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("python-dotenv not installed. Using system environment variables only.")

print("✅ Setup complete!")

python-dotenv not installed. Using system environment variables only.
✅ Setup complete!


In [3]:
class PromptEngineeringLab:
    """
    A flexible LLM wrapper that supports multiple backends.
    Automatically falls back to mock mode if no API keys are available.
    """
    
    def __init__(self, provider: str = "auto"):
        """
        Args:
            provider: "openai", "anthropic", "huggingface", "mock", or "auto"
        """
        self.provider = provider
        self.client = None
        self.model = None
        
        if provider == "auto":
            self._auto_detect_provider()
        else:
            self._initialize_provider(provider)
    
    def _auto_detect_provider(self):
        """Automatically detect which provider to use based on available API keys."""
        if os.getenv("OPENAI_API_KEY"):
            print("🔑 Detected OpenAI API key")
            self._initialize_provider("openai")
        elif os.getenv("ANTHROPIC_API_KEY"):
            print("🔑 Detected Anthropic API key")
            self._initialize_provider("anthropic")
        elif os.getenv("HUGGINGFACE_API_KEY"):
            print("🔑 Detected Hugging Face API key")
            self._initialize_provider("huggingface")
        else:
            print("⚠️  No API keys detected. Using MOCK mode.")
            print("   You'll see pre-generated responses to understand the concepts.")
            print("   To use real LLMs, set API keys as environment variables.")
            self._initialize_provider("mock")
    
    def _initialize_provider(self, provider: str):
        """Initialize the selected provider."""
        self.provider = provider
        
        if provider == "openai":
            try:
                import openai
                self.client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
                self.model = "gpt-3.5-turbo"
                print(f"✅ Using OpenAI ({self.model})")
            except Exception as e:
                print(f"❌ Failed to initialize OpenAI: {e}")
                self._initialize_provider("mock")
        
        elif provider == "anthropic":
            try:
                import anthropic
                self.client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
                self.model = "claude-3-haiku-20240307"
                print(f"✅ Using Anthropic ({self.model})")
            except Exception as e:
                print(f"❌ Failed to initialize Anthropic: {e}")
                self._initialize_provider("mock")
        
        elif provider == "huggingface":
            try:
                from transformers import pipeline
                print("Loading Hugging Face model (this may take a moment)...")
                self.client = pipeline(
                    "text-generation",
                    model="google/flan-t5-base",
                    max_length=512
                )
                self.model = "flan-t5-base"
                print(f"✅ Using Hugging Face ({self.model})")
            except Exception as e:
                print(f"❌ Failed to initialize Hugging Face: {e}")
                self._initialize_provider("mock")
        
        elif provider == "mock":
            self.client = None
            self.model = "mock-llm"
            print("✅ Using Mock LLM (pre-generated responses)")
    
    def generate(self, 
                 prompt: str, 
                 system_prompt: Optional[str] = None,
                 temperature: float = 0.7,
                 max_tokens: int = 500) -> str:
        """
        Generate a response from the LLM.
        
        Args:
            prompt: The user prompt
            system_prompt: Optional system prompt (not all models support this)
            temperature: Sampling temperature (0.0 = deterministic, 1.0 = creative)
            max_tokens: Maximum response length
        
        Returns:
            Generated text response
        """
        if self.provider == "openai":
            return self._generate_openai(prompt, system_prompt, temperature, max_tokens)
        elif self.provider == "anthropic":
            return self._generate_anthropic(prompt, system_prompt, temperature, max_tokens)
        elif self.provider == "huggingface":
            return self._generate_huggingface(prompt, system_prompt, temperature, max_tokens)
        else:
            return self._generate_mock(prompt, system_prompt)
    
    def _generate_openai(self, prompt, system_prompt, temperature, max_tokens):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    
    def _generate_anthropic(self, prompt, system_prompt, temperature, max_tokens):
        kwargs = {
            "model": self.model,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]
        }
        if system_prompt:
            kwargs["system"] = system_prompt
        
        response = self.client.messages.create(**kwargs)
        return response.content[0].text
    
    def _generate_huggingface(self, prompt, system_prompt, temperature, max_tokens):
        full_prompt = prompt
        if system_prompt:
            full_prompt = f"{system_prompt}\n\n{prompt}"
        
        result = self.client(
            full_prompt,
            max_length=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0
        )
        return result[0]['generated_text']
    
    def _generate_mock(self, prompt, system_prompt):
        """Return mock responses based on prompt patterns."""
        # Simple pattern matching for educational purposes
        prompt_lower = prompt.lower()
        
        # Vague vs specific example
        if "write about dogs" in prompt_lower and len(prompt) < 50:
            return "Dogs are great pets. They are loyal and friendly. Many people love dogs."
        elif "golden retriever" in prompt_lower and "family" in prompt_lower:
            return """Golden Retrievers are excellent family dogs for several reasons:

1. Temperament: They are known for their gentle, patient, and friendly nature
2. Intelligence: Ranked 4th in intelligence, making them easy to train
3. Size: Medium-to-large (55-75 lbs), suitable for active families
4. Exercise needs: Require 1-2 hours daily exercise
5. Good with children: Very patient and protective

However, they shed significantly and need regular grooming."""
        
        # Few-shot example responses
        elif "positive" in prompt_lower and "negative" in prompt_lower and "sentiment" in prompt_lower:
            if "terrible service" in prompt_lower:
                return "Negative"
            else:
                return "Positive"
        
        # Chain of thought
        elif "step by step" in prompt_lower or "let's think" in prompt_lower:
            if "23 * 17" in prompt or "23*17" in prompt:
                return """Let me solve this step by step:

23 * 17
= 23 * (10 + 7)
= (23 * 10) + (23 * 7)
= 230 + 161
= 391

The answer is 391."""
            else:
                return "Let me break this down step by step:\n1. First, I'll identify the key components\n2. Then, I'll analyze each part\n3. Finally, I'll synthesize a conclusion"
        
        # JSON extraction
        elif "json" in prompt_lower and ("john doe" in prompt_lower or "john smith" in prompt_lower):
            return json.dumps({
                "name": "John Smith",
                "email": "john.smith@email.com",
                "phone": "555-0123",
                "city": "New York"
            }, indent=2)
        
        # Creative vs factual
        elif "story" in prompt_lower or "creative" in prompt_lower:
            return "Once upon a time, in a land where the clouds were made of cotton candy and the rivers flowed with chocolate, there lived a curious little robot named Bolt..."
        elif "fact" in prompt_lower or "explain" in prompt_lower:
            return "Photosynthesis is the process by which plants convert light energy into chemical energy. It occurs in chloroplasts and requires sunlight, water, and carbon dioxide."
        
        # Default response
        else:
            return f"This is a mock response to demonstrate the concept. In production, the LLM would generate a real response to: '{prompt[:100]}...'"

# Initialize the lab
lab = PromptEngineeringLab(provider="anthropic")
print("\n🎮 Prompt Engineering Lab is ready!")

❌ Failed to initialize Anthropic: No module named 'anthropic'
✅ Using Mock LLM (pre-generated responses)

🎮 Prompt Engineering Lab is ready!


## 1️⃣ Specificity: Vague vs. Precise Prompts

### The Problem
Vague prompts get vague responses. The more specific you are, the better the output.

### 🎯 Challenge: Compare These Prompts

In [3]:
# VAGUE PROMPT
vague_prompt = "Write about dogs."

print("🔴 VAGUE PROMPT:")
print(f"'{vague_prompt}'\n")
print("Response:")
print("-" * 80)
response_vague = lab.generate(vague_prompt, temperature=0.7)
print(response_vague)
print("-" * 80)

🔴 VAGUE PROMPT:
'Write about dogs.'

Response:
--------------------------------------------------------------------------------
Dogs are great pets. They are loyal and friendly. Many people love dogs.
--------------------------------------------------------------------------------


In [4]:
# SPECIFIC PROMPT
specific_prompt = """Write a 150-word comparison of Golden Retrievers vs. Labrador Retrievers 
as family pets. Include:
- Temperament differences
- Exercise requirements
- Grooming needs
- Suitability for families with young children

Format as a bulleted list."""

print("🟢 SPECIFIC PROMPT:")
print(f"'{specific_prompt}'\n")
print("Response:")
print("-" * 80)
response_specific = lab.generate(specific_prompt, temperature=0.7)
print(response_specific)
print("-" * 80)

🟢 SPECIFIC PROMPT:
'Write a 150-word comparison of Golden Retrievers vs. Labrador Retrievers 
as family pets. Include:
- Temperament differences
- Exercise requirements
- Grooming needs
- Suitability for families with young children

Format as a bulleted list.'

Response:
--------------------------------------------------------------------------------
Golden Retrievers are excellent family dogs for several reasons:

1. Temperament: They are known for their gentle, patient, and friendly nature
2. Intelligence: Ranked 4th in intelligence, making them easy to train
3. Size: Medium-to-large (55-75 lbs), suitable for active families
4. Exercise needs: Require 1-2 hours daily exercise
5. Good with children: Very patient and protective

However, they shed significantly and need regular grooming.
--------------------------------------------------------------------------------


### 💡 Key Takeaways

**Specificity includes:**
- **Length**: "Write 150 words" vs. "Write something"
- **Format**: "Bulleted list" vs. unspecified
- **Content requirements**: Specific topics to cover
- **Style/Tone**: "Professional" vs. "Casual" vs. "For children"
- **Audience**: "For experts" vs. "For beginners"

### ✏️ YOUR TURN: Practice Specificity

Modify the prompt below to be more specific. Try to get a better response!

In [5]:
# TODO: Modify this prompt to be more specific
your_prompt = "Tell me about climate change."

# Test your prompt
print("YOUR PROMPT:")
print(f"'{your_prompt}'\n")
print("Response:")
print("-" * 80)
your_response = lab.generate(your_prompt, temperature=0.7)
print(your_response)
print("-" * 80)

# Tips:
# - Specify word count or length
# - Define the audience (students, policymakers, etc.)
# - Request specific format (bullet points, table, etc.)
# - Focus on specific aspects (causes, solutions, impacts, etc.)

YOUR PROMPT:
'Tell me about climate change.'

Response:
--------------------------------------------------------------------------------
This is a mock response to demonstrate the concept. In production, the LLM would generate a real response to: 'Tell me about climate change....'
--------------------------------------------------------------------------------


## 2️⃣ Few-Shot Learning: Teaching by Example

### The Technique
Instead of just describing what you want, **show examples** of input-output pairs. This is called "few-shot learning."

- **Zero-shot**: No examples, just instructions
- **One-shot**: One example
- **Few-shot**: Multiple examples (usually 2-5)

### Example: Sentiment Classification

In [6]:
# ZERO-SHOT
zero_shot = """Classify the sentiment of this review as Positive or Negative:

Review: "The food was cold and the service was terrible."
Sentiment:"""

print("🔵 ZERO-SHOT (no examples):")
print(zero_shot)
print("\nResponse:", lab.generate(zero_shot, temperature=0))
print("=" * 80)

🔵 ZERO-SHOT (no examples):
Classify the sentiment of this review as Positive or Negative:

Review: "The food was cold and the service was terrible."
Sentiment:

Response: Positive


In [7]:
# FEW-SHOT
few_shot = """Classify the sentiment of reviews as Positive or Negative.

Review: "Amazing food! The pasta was perfectly cooked and the staff was wonderful."
Sentiment: Positive

Review: "Overpriced and bland. Would not recommend."
Sentiment: Negative

Review: "Best restaurant in town! Can't wait to come back."
Sentiment: Positive

Review: "The food was cold and the service was terrible."
Sentiment:"""

print("🟢 FEW-SHOT (3 examples):")
print(few_shot)
print("\nResponse:", lab.generate(few_shot, temperature=0))
print("=" * 80)

🟢 FEW-SHOT (3 examples):
Classify the sentiment of reviews as Positive or Negative.

Review: "Amazing food! The pasta was perfectly cooked and the staff was wonderful."
Sentiment: Positive

Review: "Overpriced and bland. Would not recommend."
Sentiment: Negative

Review: "Best restaurant in town! Can't wait to come back."
Sentiment: Positive

Review: "The food was cold and the service was terrible."
Sentiment:

Response: Positive


### 💡 When to Use Few-Shot

- **Custom formats**: When you need a specific output structure
- **Ambiguous tasks**: When instructions alone aren't clear
- **Style matching**: When you want a particular tone or style
- **Complex patterns**: When the task has subtle rules

### ✏️ YOUR TURN: Create a Few-Shot Prompt

Create a few-shot prompt to extract product names and prices from messy text.

In [ ]:
# TODO: Add 2-3 examples before the test case
your_few_shot = """Extract the product name and price.

Text: "Check out our new iPhone 15 Pro for just $999!"
Product: iPhone 15 Pro
Price: $999

Text: "Get the Samsung Galaxy Tab S9 today - only $799.99 while supplies last!"
Product: Samsung Galaxy Tab S9
Price: $799.99

Text: "Limited offer: MacBook Air M2 now available at an amazing price of $1,199!!!"
Product:"""

print("YOUR FEW-SHOT PROMPT:")
print(your_few_shot)
print("\nResponse:")
response = lab.generate(your_few_shot, temperature=0)
print(response)

## 3️⃣ Chain-of-Thought: Make the Model "Think"

### The Technique
Ask the model to **show its reasoning process** before giving the final answer. This dramatically improves performance on complex tasks.

### Example: Math Problem

In [ ]:
# WITHOUT CHAIN-OF-THOUGHT
direct_prompt = "What is 23 * 17?"

print("🔴 DIRECT (no reasoning):")
print(direct_prompt)
print("\nResponse:")
print(lab.generate(direct_prompt, temperature=0))
print("=" * 80)

In [ ]:
# WITH CHAIN-OF-THOUGHT
cot_prompt = """What is 23 * 17?

Let's solve this step by step:"""

print("🟢 CHAIN-OF-THOUGHT (with reasoning):")
print(cot_prompt)
print("\nResponse:")
print(lab.generate(cot_prompt, temperature=0))
print("=" * 80)

### Magic Phrases for Chain-of-Thought

- "Let's think step by step."
- "Let's work through this systematically."
- "First, let's identify... Then, let's..."
- "Show your reasoning before the final answer."

### ✏️ YOUR TURN: Chain-of-Thought Reasoning

Apply chain-of-thought to this logic puzzle:

In [ ]:
puzzle = """A farmer has 17 sheep. All but 9 die. How many sheep are left?

Think through this carefully:"""

print("LOGIC PUZZLE:")
print(puzzle)
print("\nResponse:")
print(lab.generate(puzzle, temperature=0))

# Try modifying the prompt to improve the reasoning!

## 4️⃣ System Prompts vs. User Prompts

### The Difference

- **System Prompt**: Sets the overall behavior, role, and constraints (the "identity")
- **User Prompt**: The specific task or question

Think of it like:
- **System**: "You are a professional editor at a major newspaper."
- **User**: "Edit this article for grammar."

### Example: Customer Service Bot

In [ ]:
# WITHOUT SYSTEM PROMPT
user_only = "A customer says: 'Your product broke after one day! I want a refund NOW!'"

print("🔴 USER PROMPT ONLY:")
print(user_only)
print("\nResponse:")
print(lab.generate(user_only, temperature=0.7))
print("=" * 80)

In [ ]:
# WITH SYSTEM PROMPT
system = """You are a professional customer service representative. Always:
- Be empathetic and understanding
- Acknowledge the customer's frustration
- Offer specific solutions
- Use a calm, professional tone
- Keep responses under 100 words"""

user = "A customer says: 'Your product broke after one day! I want a refund NOW!'"

print("🟢 WITH SYSTEM PROMPT:")
print(f"System: {system}")
print(f"\nUser: {user}")
print("\nResponse:")
print(lab.generate(user, system_prompt=system, temperature=0.7))
print("=" * 80)

### 💡 System Prompt Best Practices

1. **Define the role**: "You are a..."
2. **Set constraints**: "Keep responses under X words", "Never mention..."
3. **Specify tone**: "Professional", "Friendly", "Technical"
4. **List rules**: What to always do, what to never do
5. **Give context**: Background information that applies to all interactions

### ✏️ YOUR TURN: Create a System Prompt

Design a system prompt for a coding tutor bot:

In [ ]:
# TODO: Create an effective system prompt for a coding tutor
your_system_prompt = """You are a helpful coding tutor."""

# Test it with a student question
student_question = "I don't understand what a for loop does. Can you help?"

print("YOUR SYSTEM PROMPT:")
print(your_system_prompt)
print(f"\nStudent Question: {student_question}")
print("\nResponse:")
print(lab.generate(student_question, system_prompt=your_system_prompt, temperature=0.7))

# Tips for improvement:
# - Specify teaching style (Socratic, direct, etc.)
# - Set expertise level expectations
# - Include rules about code examples
# - Define how to handle difficult questions

## 🏆 CHALLENGE 1: Structured Data Extraction

### The Task
Extract structured information from messy, unstructured text and output it as valid JSON.

### The Goal
Create a prompt that reliably extracts:
- Name
- Email
- Phone number
- City

...from messy text like emails or messages.

In [ ]:
# Sample messy text
messy_text = """Hey there! My name is John Smith and I'm reaching out about the job posting.
You can reach me at john.smith@email.com or call me at 555-0123. I'm currently living in 
New York City and would love to discuss this opportunity. Looking forward to hearing from you!"""

# TODO: Create a prompt that extracts structured data as JSON
# Hint: Use few-shot examples, be specific about JSON format, and use chain-of-thought if needed
your_extraction_prompt = f"""Extract the contact information from this text:

{messy_text}
"""

print("YOUR PROMPT:")
print(your_extraction_prompt)
print("\n" + "=" * 80)
print("RESPONSE:")
result = lab.generate(your_extraction_prompt, temperature=0)
print(result)
print("=" * 80)

# Try to parse it as JSON
try:
    import json
    # Try to find JSON in the response
    if '{' in result:
        json_start = result.index('{')
        json_end = result.rindex('}') + 1
        json_str = result[json_start:json_end]
        parsed = json.loads(json_str)
        print("\n✅ Successfully parsed as JSON:")
        print(json.dumps(parsed, indent=2))
    else:
        print("\n⚠️  No JSON found in response. Try being more specific about the format!")
except Exception as e:
    print(f"\n❌ Could not parse as JSON: {e}")
    print("Try improving your prompt to ensure valid JSON output!")

### 💡 Tips for Better Structured Extraction

1. **Provide a JSON template** in your prompt
2. **Use few-shot examples** showing input → JSON output
3. **Specify field requirements**: "Use null if not found"
4. **Request only valid JSON**: "Output ONLY the JSON, no additional text"
5. **Validate and iterate**: Test on multiple examples

## 🏆 CHALLENGE 2: Creative vs. Factual Generation

### The Task
Learn to control the balance between creativity and factual accuracy using:
- Temperature settings
- Prompt instructions
- System prompts

### Temperature Scale
- **0.0 - 0.3**: Deterministic, factual, consistent
- **0.4 - 0.7**: Balanced creativity and coherence
- **0.8 - 1.0**: Highly creative, unpredictable, diverse

In [ ]:
# FACTUAL TASK: Explaining a scientific concept
factual_prompt = "Explain how photosynthesis works in 3-4 sentences."

print("📚 FACTUAL TASK (Temperature = 0.0):")
print(factual_prompt)
print("\nResponse:")
print(lab.generate(factual_prompt, temperature=0.0))
print("=" * 80)

In [ ]:
# CREATIVE TASK: Story generation
creative_prompt = "Write the opening paragraph of a story about a robot who discovers emotions."

print("🎨 CREATIVE TASK (Temperature = 0.9):")
print(creative_prompt)
print("\nResponse:")
print(lab.generate(creative_prompt, temperature=0.9))
print("=" * 80)

### ✏️ YOUR TURN: Control Creativity

Experiment with the same prompt at different temperatures:

In [ ]:
base_prompt = "Describe a sunset in one sentence."

temperatures = [0.0, 0.5, 1.0]

for temp in temperatures:
    print(f"\n🌡️  Temperature = {temp}:")
    print("-" * 80)
    response = lab.generate(base_prompt, temperature=temp)
    print(response)

print("\n" + "=" * 80)
print("Notice how the responses change with temperature!")

## 🏆 FINAL CHALLENGE: Prompt Engineering Competition

### The Ultimate Test

**Scenario**: You're building a chatbot for a restaurant. It needs to:
1. Take customer orders
2. Answer menu questions
3. Handle complaints professionally
4. Upsell appropriately (but not annoyingly)
5. Output orders in a structured JSON format

**Your Mission**: Create the perfect system prompt + user interaction.

### Sample Menu
```
Pizzas: Margherita ($12), Pepperoni ($14), Veggie Supreme ($15)
Sides: Garlic Bread ($5), Caesar Salad ($7), Wings ($9)
Drinks: Soda ($2), Beer ($6), Wine ($8)
```

In [ ]:
# TODO: Create your restaurant chatbot system prompt
restaurant_system = """You are a friendly restaurant order-taking assistant."""

# Test scenarios
test_cases = [
    "Hi, I'd like to order a pizza.",
    "What vegetarian options do you have?",
    "I'll take a Margherita pizza. That's all.",
    "My last order was cold! I'm not happy.",
]

print("🍕 RESTAURANT CHATBOT TEST")
print("=" * 80)
print("SYSTEM PROMPT:")
print(restaurant_system)
print("=" * 80)

for i, customer_message in enumerate(test_cases, 1):
    print(f"\n[Test {i}]")
    print(f"Customer: {customer_message}")
    print("Bot: ", end="")
    response = lab.generate(customer_message, system_prompt=restaurant_system, temperature=0.7)
    print(response)
    print("-" * 80)

print("\n💡 How well did your chatbot perform?")
print("Iterate on the system prompt to improve it!")

### 🎯 Scoring Your Chatbot

Rate your chatbot on these criteria:

1. **Friendliness** (1-5): Is it welcoming and polite?
2. **Accuracy** (1-5): Does it reference the correct menu items and prices?
3. **Upselling** (1-5): Does it suggest add-ons naturally?
4. **Complaint Handling** (1-5): Professional and empathetic?
5. **Structure** (1-5): Could it output a clean JSON order?

**Total Score**: _____ / 25

Share your system prompt with classmates and compare scores!

## 🎓 Summary: Prompt Engineering Principles

### The Golden Rules

1. **Be Specific**
   - Clear instructions > vague requests
   - Specify format, length, style, and constraints

2. **Show Examples (Few-Shot)**
   - Demonstrate the pattern you want
   - 2-5 examples is often the sweet spot

3. **Use Chain-of-Thought**
   - Ask for step-by-step reasoning
   - Especially powerful for complex tasks

4. **Leverage System Prompts**
   - Set the identity and rules once
   - Keep user prompts focused on the task

5. **Control Temperature**
   - Low (0-0.3) for factual tasks
   - High (0.7-1.0) for creative tasks

6. **Iterate and Test**
   - First prompt is rarely perfect
   - Test on multiple examples
   - Refine based on failures

### Advanced Techniques (For Further Study)

- **Self-consistency**: Generate multiple responses and vote
- **Prompt chaining**: Break complex tasks into steps
- **Retrieval-augmented generation (RAG)**: Provide relevant context
- **Constitutional AI**: Give principles to guide behavior

### 📚 Resources

- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Prompt Library](https://docs.anthropic.com/claude/prompt-library)
- [Learn Prompting](https://learnprompting.org/)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)

### 🚀 Next Steps

1. Practice on real tasks from your work/studies
2. Build a personal prompt library
3. Experiment with different models
4. Join prompt engineering communities
5. Stay updated - this field evolves rapidly!

## 🎮 Bonus: Create Your Own Prompt Challenge

Use this cell to experiment freely!

In [ ]:
# Your custom system prompt
my_system = """
# Add your system prompt here
"""

# Your custom user prompt
my_prompt = """
# Add your user prompt here
"""

# Test it!
if my_prompt.strip():
    response = lab.generate(
        my_prompt, 
        system_prompt=my_system if my_system.strip() else None,
        temperature=0.7
    )
    print("Response:")
    print("=" * 80)
    print(response)
    print("=" * 80)
else:
    print("Add your prompt above and run this cell!")

---

## 🎉 Congratulations!

You've completed the Prompt Engineering Playground! You now know:

✅ How to write specific, effective prompts
✅ When and how to use few-shot learning
✅ How to leverage chain-of-thought reasoning
✅ The difference between system and user prompts
✅ How to control creativity with temperature
✅ How to extract structured data from text

**Remember**: Prompt engineering is part art, part science. The best way to improve is to practice, experiment, and learn from failures.

Happy prompting! 🚀